# Building a TethysDash plugin — the example plot

Everything on a TethysDash dashboard is drawn by a **plugin**: an installable
Python package that the app discovers on its own. This notebook builds the
simplest useful one from scratch — a line chart — and then adds an argument and a
progress bar to it.

The question being answered is:

> **What is the smallest amount of Python that puts a new chart in the
> visualization picker?**

The answer is a class with four attributes and one method. Everything else is
optional.

**What to take away from it**

- what `run()` is actually required to return, and how to check it yourself
- how declaring `args` gets you interface controls without writing a form
- why `get_arg()` is safer than reading the argument off `self`
- what registering a plugin means, and why it needs no change to TethysDash

**Workflow**

1. Get the data and build the chart with plotly, as you would anywhere
2. Look at what `to_json()` produces — that *is* the plugin's return value
3. Wrap it in a plugin class and run it here, without a server
4. Add an argument, and see where the interface control comes from
5. Add progress updates
6. Register it so the app can find it

A real plugin imports its base class from TethysDash:

```python
from tethysapp.tethysdash.plugin_helpers import TethysDashPlugin
```

This notebook defines a **stand-in** with the same contract instead, so every cell
runs on a laptop with nothing but plotly installed. That one import line is the
only difference between what is here and a real plugin file.

In [1]:
!pip install plotly


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import json

import plotly.express as px
import plotly.graph_objects as go

VALID_TYPES = ["plotly", "table", "image", "card", "text", "variable_input",
               "map", "map_layer", "custom", "imageCollection"]

class TethysDashPlugin:
    """Teaching stand-in for the real base class, with the same contract.

    Mirrors what the real one does at construction: it validates the four
    required attributes, rejects an unknown `type`, and exposes the supplied
    arguments through both `get_arg()` and attribute access.

    `send_update()` here just prints. The real one publishes over a WebSocket and
    needs the request context the app attaches when it invokes a plugin, so it
    cannot be called outside a running server -- which is exactly why this
    notebook uses a stand-in rather than importing the real class.
    """

    args = {}

    def __init__(self, **kwargs):
        for attr in ("name", "type", "label", "group"):
            if getattr(self, attr, None) in (None, ""):
                raise ValueError(f"Plugin must have a {attr} attribute defined.")
        if self.type not in VALID_TYPES:
            raise ValueError(
                f"Plugin type '{self.type}' is not valid. "
                f"Must be one of: {', '.join(VALID_TYPES)}"
            )
        self.received_args = dict(kwargs)
        for key, value in kwargs.items():
            setattr(self, key, value)

    def get_arg(self, name, default=None):
        return self.received_args.get(name, default)

    def send_update(self, message, percentage_complete=None, layer_id=None):
        pct = "" if percentage_complete is None else f" [{percentage_complete}%]"
        print(f"  progress{pct}: {message}")

print(f"stand-in ready | {len(VALID_TYPES)} valid plugin types")

stand-in ready | 10 valid plugin types


## 1. The chart, built the ordinary way

Nothing about this step is TethysDash-specific. It is the plot you would write in
any notebook — which is the point: a plugin is the code you already have, wrapped
so the app can call it.

In [3]:
df = px.data.gapminder()
print(f"{len(df):,} rows, {df.country.nunique()} countries, "
      f"{df.year.min()}-{df.year.max()}")
df.head(3)

1,704 rows, 142 countries, 1952-2007


,country,continent,year,lifeExp,pop,gdpPercap,iso_alpha,iso_num
0,Afghanistan,Asia,1952,28.801,8425333,779.445314,AFG,4
1,Afghanistan,Asia,1957,30.332,9240934,820.853030,AFG,4
2,Afghanistan,Asia,1962,31.997,10267083,853.100710,AFG,4


In [4]:
asia = df.query("continent == 'Asia'")
fig = px.line(asia, x="year", y="lifeExp", color="country", symbol="country")
fig

## 2. What `run()` has to return

A plugin declares `type = "plotly"`, and that choice decides the shape of its
return value. For `plotly` the contract is the figure as a **plain dictionary** —
exactly what `to_json()` produces.

Look at the keys rather than the contents:

In [5]:
payload = json.loads(fig.to_json())

print("top-level keys:", list(payload))
print(f"  data:   {len(payload['data'])} traces")
print(f"  layout: {len(payload['layout'])} keys -> {list(payload['layout'])[:6]}...")
print(f"\nserialised size: {len(fig.to_json()) / 1024:.0f} KB")

top-level keys: ['data', 'layout']
  data:   33 traces
  layout: 5 keys -> ['template', 'xaxis', 'yaxis', 'legend', 'margin']...

serialised size: 25 KB


That dictionary is the whole interface. The plugin does not draw anything and
does not touch the browser — it returns data, and the frontend renders it.

Which means the contract is checkable without a server. Round-trip the payload
back into a figure: if it draws, a real dashboard would draw the same thing.

In [6]:
go.Figure(payload)

## 3. The minimal plugin

Four required attributes and one method:

| attribute | what it does |
|---|---|
| `name` | the install and driver name — must match the entry point |
| `group` | groups the plugin in the visualization picker |
| `label` | the name shown in the app |
| `type` | picks the renderer, and therefore dictates what `run()` returns |

Leaving any of them out raises at construction, so a broken plugin fails loudly
rather than half-appearing in the app.

In [7]:
class PlotExample(TethysDashPlugin):
    name = "plot_example"
    group = "Example"
    label = "Example Plot"
    type = "plotly"

    def run(self):
        frame = px.data.gapminder().query("continent == 'Asia'")
        figure = px.line(frame, x="year", y="lifeExp",
                         color="country", symbol="country")
        return json.loads(figure.to_json())


plugin = PlotExample()
result = plugin.run()
print(f"run() returned {type(result).__name__} with keys {list(result)}")
print(f"  {len(result['data'])} traces")

run() returned dict with keys ['data', 'layout']
  33 traces


Instantiating and calling `run()` is also how you test a plugin — no server, no
dashboard, no browser. If `run()` returns the right shape, the visualization
works.

In [8]:
# What happens when a required attribute is missing.
class Broken(TethysDashPlugin):
    name = "broken"
    type = "plotly"
    label = "Broken"
    # group is missing

try:
    Broken()
except ValueError as err:
    print(f"ValueError: {err}")

ValueError: Plugin must have a group attribute defined.


## 4. Adding an argument

Declaring `args` is what produces the interface controls. The dashboard author
never sees a form you wrote — TethysDash generates the input from the type you
declare, and the label from the argument's own name.

`{"continent": "text"}` becomes a text box labelled **Continent**.

In [9]:
class PlotByContinent(TethysDashPlugin):
    name = "plot_by_continent"
    group = "Example"
    label = "Plot by Continent"
    type = "plotly"
    args = {"continent": "text"}      # -> one text input, labelled "Continent"

    def run(self):
        continent = self.get_arg("continent", "Asia")
        frame = px.data.gapminder().query(f"continent == '{continent}'")
        figure = px.line(frame, x="year", y="lifeExp",
                         color="country", symbol="country")
        figure.update_layout(title=f"Life expectancy — {continent}")
        return json.loads(figure.to_json())


# The app passes the configured values in; here we pass them by hand.
for continent in ("Europe", "Africa"):
    out = PlotByContinent(continent=continent).run()
    print(f"{continent:<8} {len(out['data']):>2} traces  "
          f"title={out['layout']['title']['text']!r}")

Europe   30 traces  title='Life expectancy — Europe'


Africa   52 traces  title='Life expectancy — Africa'


In [10]:
go.Figure(PlotByContinent(continent="Europe").run())

### Read arguments with `get_arg()`, not off `self`

The example on the slides uses `self.continent`, and for a simple name that works
— the framework sets each supplied argument as an attribute. But **`get_arg()` is
the one to prefer**, for two reasons:

- **Nested arguments have dotted names**, like `transect_location.location`.
  Python cannot resolve a dotted attribute, so `self.transect_location.location`
  fails — quietly, in the worst cases.
- **`get_arg()` takes a default.** An argument the author left blank simply is not
  there, so attribute access raises `AttributeError` while
  `get_arg("continent", "Asia")` carries on.

`args` also cannot use the names of the plugin's own properties — `type`, `label`,
`group`, `tags` and the rest are reserved, and colliding with one raises at
construction.

## 5. Progress updates

A plugin that takes a few seconds should say so. `send_update()` streams a message
— and optionally a percentage — to the dashboard item over a WebSocket, which
turns a blank tile into a progress bar.

Here the stand-in prints instead, so you can see the sequence.

One practical note: the real `send_update()` needs the request context the app
attaches when it calls a plugin, so it only works inside a running server. Calling
it from a script raises `AttributeError`. That is not a problem in practice — it
just means progress reporting is the one part of a plugin you cannot exercise
outside the app.

In [11]:
class PlotWithProgress(TethysDashPlugin):
    name = "plot_with_progress"
    group = "Example"
    label = "Plot with Progress"
    type = "plotly"
    args = {"continent": "text"}

    def run(self):
        continent = self.get_arg("continent", "Asia")

        self.send_update("Gathering data from plotly", percentage_complete=25)
        frame = px.data.gapminder().query(f"continent == '{continent}'")

        self.send_update("Plotting data", percentage_complete=75)
        figure = px.line(frame, x="year", y="lifeExp",
                         color="country", symbol="country")

        self.send_update("Done", percentage_complete=100)
        return json.loads(figure.to_json())


print("running PlotWithProgress(continent='Oceania')")
out = PlotWithProgress(continent="Oceania").run()
print(f"-> {len(out['data'])} traces")

running PlotWithProgress(continent='Oceania')
  progress [25%]: Gathering data from plotly
  progress [75%]: Plotting data
  progress [100%]: Done
-> 2 traces


For the flood plugins in exercises 2 and 3 this is not cosmetic. Sampling depth
onto 5,000 buildings takes a few seconds, and without progress the tile looks
broken rather than busy.

## 6. Registering it

The plugin exists, but the app cannot see it yet. Registration is a single entry
point in the package's `pyproject.toml` — TethysDash never gets edited.

```toml
[project.entry-points."intake.drivers"]
plot_example = "my_plugin.source:PlotExample"
```

Then:

```bash
pip install .        # into the environment TethysDash runs in
```

Restart the app and the plugin appears in the **Visualization Type** dropdown,
under whatever `group` it declared. Six things happen, none of them inside
TethysDash:

1. The entry point is declared in `pyproject.toml` (or `setup.py`)
2. `pip install` puts the package in the TethysDash environment
3. Intake auto-registers it — `open_plot_example` becomes callable
4. The plugin shows up in the visualization picker
5. A `static/` folder of thumbnails makes it recognisable among many
6. Testing is instantiating the class and calling `run()`, as above

The institutional consequence is worth stating plainly: **installing a plugin is
installing a Python package.** It needs no new procedure, no change to TethysDash,
and nothing that blocks upgrading the platform later.

### The optional properties

| property | default | what it does |
|---|---|---|
| `args` | `{}` | argument schema → generated inputs |
| `tags` | `[]` | search and discovery |
| `description` | `""` | shown to users beside the selection |
| `restricted` | `False` | limits the plugin to permitted users |
| `loading_icon` | `True` | spinner while the plugin runs |
| `attribution` | `""` | data-source credit drawn on the widget |
| `dynamic_map_layer` | `False` | enables `fetch_features()` for live map layers |

Two matter in an institutional setting: `restricted`, which gates a plugin behind
permissions, and `attribution`, which prints the data-source credit automatically
— often a formal requirement when publishing someone else's data.

### Things to try

- Change `type` to something invalid and see the error. Then try `"table"` and
  make `run()` return `{"title": ..., "data": [ ... ]}` instead — the same class,
  a different renderer.
- Declare `args = {"continent": "text", "year": "number"}` and filter on both.
  What would the interface show?
- Try `args = {"type": "text"}` and read the error. Why is that name reserved?
- Remove the `get_arg` default and instantiate with no `continent`. Compare the
  failure with what `self.continent` would have done.
- The chart is redrawn from scratch on every call. Which parts of `run()` would
  you cache if the data came from a slow source rather than from plotly?